# ML-04 Optional — Feature Vector & Leakage Deep Dive

**Lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026  

This optional notebook extends the required ML-04 data contract. It builds the same five decision-time-safe features, audits their availability and missingness, then deliberately attacks the feature set with label-derived and future-window information. The final retained feature list is explicitly checked to contain no leakage.

> **Important:** the Hugging Face token is read from Colab Secrets as `HF_TOKEN`. Never paste it into a cell or commit it.


## 0. What this notebook is proving

The required notebook established the contract. This deeper version asks four stronger questions:

1. Can the five features be rebuilt from the March decision window only?
2. Are the resulting rows at the intended content-client decision grain?
3. Does deliberately exposing the April outcome make the score collapse into an obviously invalid near-perfect result?
4. After removing the trap, is the final feature list demonstrably free of future/label-derived columns?

The April outcome is used only to create the evaluation label and to demonstrate leakage. It is never retained as a production feature.


In [1]:
%pip -q install duckdb scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add a Read token in Colab Secrets with the name HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

MARCH_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

print("Warehouse connection configured.")
print("Decision window: 2026-03")
print("Outcome window: 2026-04")


Warehouse connection configured.
Decision window: 2026-03
Outcome window: 2026-04


## 1. Resolve the physical warehouse columns

Do not guess column names. Inspect the March partition and resolve the semantic fields used by this lane.


In [2]:
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MARCH_REL}')").df()
display(schema[["column_name", "column_type"]])

available = set(schema["column_name"])

def resolve(candidates, label):
    for name in candidates:
        if name in available:
            return name
    raise KeyError(f"Could not resolve {label}. Available columns: {sorted(available)}")

DATE_COL = resolve(["report_date"], "report date")
CLIENT_COL = resolve(["client_hash_id", "client_id"], "client key")
CONTENT_COL = resolve(["content_hash_id", "content_id"], "content key")
GSC_AVAIL_COL = resolve(["gsc_data_available"], "GSC availability")
IMP_COL = resolve(["gsc_impressions", "impressions"], "GSC impressions")
CLICK_COL = resolve(["gsc_clicks", "clicks"], "GSC clicks")
POSITION_COL = resolve(["gsc_avg_position", "avg_position"], "GSC position")

resolved = pd.DataFrame({
    "semantic_field": ["date", "client", "content", "GSC availability", "GSC impressions", "GSC clicks", "GSC position"],
    "physical_column": [DATE_COL, CLIENT_COL, CONTENT_COL, GSC_AVAIL_COL, IMP_COL, CLICK_COL, POSITION_COL],
})
display(resolved)


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


,semantic_field,physical_column
0,date,report_date
1,client,client_hash_id
2,content,content_hash_id
3,GSC availability,gsc_data_available
4,GSC impressions,gsc_impressions
5,GSC clicks,gsc_clicks
6,GSC position,gsc_avg_position


## 2. Verification: source grain and March window

The raw fact table grain is one content-client-date row. The decision frame will later aggregate this to one row per content-client pair.


In [3]:
grain_check = con.sql(f"""
SELECT {CLIENT_COL} AS client_hash_id,
       {CONTENT_COL} AS content_hash_id,
       {DATE_COL} AS report_date,
       COUNT(*) AS row_count
FROM read_parquet('{MARCH_REL}')
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
display(grain_check)
assert grain_check.empty

window_check = con.sql(f"""
SELECT COUNT(*) AS march_rows,
       MIN({DATE_COL}) AS min_report_date,
       MAX({DATE_COL}) AS max_report_date
FROM read_parquet('{MARCH_REL}')
""").df()
display(window_check)
assert str(window_check.loc[0, "min_report_date"])[:10] == "2026-03-01"
assert str(window_check.loc[0, "max_report_date"])[:10] == "2026-03-31"
print("PASS: raw grain is unique at content-client-date and March spans 2026-03-01 through 2026-03-31.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


,march_rows,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


PASS: raw grain is unique at content-client-date and March spans 2026-03-01 through 2026-03-31.


## 3. Availability audit (`IS TRUE`)

Rows without usable GSC data are not treated as genuine zero activity. The availability flag is therefore part of the population rule.


In [4]:
availability = con.sql(f"""
SELECT COUNT(*) AS all_march_rows,
       COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS TRUE) AS gsc_available_rows,
       COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS NOT TRUE) AS gsc_not_available_rows,
       COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS NULL) AS gsc_null_flag_rows
FROM read_parquet('{MARCH_REL}')
""").df()
display(availability)
print(f"GSC-available March rows: {int(availability.loc[0, 'gsc_available_rows']):,} of {int(availability.loc[0, 'all_march_rows']):,} total rows.")
assert int(availability.loc[0, "gsc_available_rows"]) > 0
print("PASS: availability was explicitly evaluated with IS TRUE.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,gsc_available_rows,gsc_not_available_rows,gsc_null_flag_rows
0,9841378,3611061,6230317,0


GSC-available March rows: 3,611,061 of 9,841,378 total rows.
PASS: availability was explicitly evaluated with IS TRUE.


## 4. Build the decision frame

The production-safe feature vector contains exactly five March-only features:

- `march_impressions`
- `march_clicks`
- `march_ctr_pct`
- `march_avg_position`
- `march_impression_days`

The label is `future_decline_label`: April impressions are more than 20% below March impressions, conditional on positive March impressions. April is the future outcome window, so it is not part of the feature vector.


In [5]:
feature_sql = f"""
WITH march AS (
    SELECT
        {CLIENT_COL} AS client_hash_id,
        {CONTENT_COL} AS content_hash_id,
        SUM({IMP_COL}) AS march_impressions,
        SUM({CLICK_COL}) AS march_clicks,
        CASE WHEN SUM({IMP_COL}) > 0
             THEN 100.0 * SUM({CLICK_COL}) / SUM({IMP_COL})
             ELSE NULL END AS march_ctr_pct,
        SUM(CASE WHEN {IMP_COL} > 0 AND {POSITION_COL} > 0
                 THEN {IMP_COL} * {POSITION_COL} ELSE 0 END)
        / NULLIF(SUM(CASE WHEN {IMP_COL} > 0 AND {POSITION_COL} > 0
                          THEN {IMP_COL} ELSE 0 END), 0) AS march_avg_position,
        COUNT(DISTINCT CASE WHEN {IMP_COL} > 0 THEN {DATE_COL} END) AS march_impression_days
    FROM read_parquet('{MARCH_REL}')
    WHERE {GSC_AVAIL_COL} IS TRUE
    GROUP BY 1,2
),
april AS (
    SELECT
        {CLIENT_COL} AS client_hash_id,
        {CONTENT_COL} AS content_hash_id,
        SUM({IMP_COL}) AS april_impressions
    FROM read_parquet('{APRIL_REL}')
    WHERE {GSC_AVAIL_COL} IS TRUE
    GROUP BY 1,2
)
SELECT m.*, a.april_impressions,
       CASE WHEN m.march_impressions > 0
                 AND a.april_impressions < 0.80 * m.march_impressions
            THEN 1 ELSE 0 END AS future_decline_label
FROM march m
INNER JOIN april a USING (client_hash_id, content_hash_id)
"""

feature_frame = con.sql(feature_sql).df()
feature_frame["april_impressions"] = feature_frame["april_impressions"].fillna(0)

clean_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_impression_days",
]

assert len(clean_features) == 5
assert set(clean_features).issubset(feature_frame.columns)
assert feature_frame["future_decline_label"].isin([0,1]).all()

print(f"Decision rows: {len(feature_frame):,}")
print(f"Future decline rate: {feature_frame['future_decline_label'].mean():.3f}")
display(feature_frame[clean_features + ["future_decline_label"]].head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision rows: 158,549
Future decline rate: 0.478


,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,future_decline_label
0,77.0,0.0,0.000000,4.742857,24,1
1,10849.0,22.0,0.202784,8.049866,31,0
2,61.0,0.0,0.000000,6.410714,27,0
3,705.0,1.0,0.141844,5.872159,31,1
4,50.0,0.0,0.000000,15.276596,21,1
5,2099.0,1.0,0.047642,2.632206,31,1
6,288.0,0.0,0.000000,10.526132,30,1
7,3535.0,25.0,0.707214,3.151344,31,0
8,1558.0,3.0,0.192555,5.247112,31,1
9,2021.0,0.0,0.000000,7.853538,31,1


## 5. Feature availability and missingness audit

Every retained feature must be knowable at the March decision point. Missingness is measured rather than silently assumed to mean zero.


In [6]:
feature_audit = pd.DataFrame({
    "feature": clean_features,
    "dtype": [str(feature_frame[c].dtype) for c in clean_features],
    "missing_rows": [int(feature_frame[c].isna().sum()) for c in clean_features],
    "missing_pct": [100.0 * feature_frame[c].isna().mean() for c in clean_features],
})
display(feature_audit)

feature_notes = pd.DataFrame({
    "feature": clean_features,
    "available_when": [
        "By the March decision point; accumulated GSC impressions through March 31.",
        "By the March decision point; accumulated GSC clicks through March 31.",
        "By the March decision point; computed only from March clicks and impressions.",
        "By the March decision point; computed only from March GSC position observations.",
        "By the March decision point; counts March dates with positive observed impressions.",
    ]
})
display(feature_notes)


,feature,dtype,missing_rows,missing_pct
0,march_impressions,float64,0,0.000000
1,march_clicks,float64,0,0.000000
2,march_ctr_pct,float64,0,0.000000
3,march_avg_position,float64,759,0.478716
4,march_impression_days,int64,0,0.000000


,feature,available_when
0,march_impressions,By the March decision point; accumulated GSC i...
1,march_clicks,By the March decision point; accumulated GSC c...
2,march_ctr_pct,By the March decision point; computed only fro...
3,march_avg_position,By the March decision point; computed only fro...
4,march_impression_days,By the March decision point; counts March date...


## 6. Leakage attack A — direct label-derived feature

This is the strongest possible trap: copy the future label itself into a fake feature. A near-perfect score is expected because the model has been handed the answer. This score is evidence that the leakage test works, not evidence of model quality.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_df = feature_frame[clean_features + ["future_decline_label"]].copy()
X = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(model_df[clean_features]),
    columns=clean_features,
    index=model_df.index,
)
y = model_df["future_decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

def fit_auc(X_train, X_test, y_train, y_test):
    model = RandomForestClassifier(
        n_estimators=150, random_state=42, n_jobs=-1, class_weight="balanced"
    )
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:,1])

clean_auc = fit_auc(X_train, X_test, y_train, y_test)

leaky = X.copy()
leaky["leaked_decline_score"] = y
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    leaky, y, test_size=0.20, random_state=42, stratify=y
)
leaky_auc = fit_auc(Xl_train, Xl_test, yl_train, yl_test)

print(f"Clean-feature ROC-AUC: {clean_auc:.3f}")
print(f"Leaky ROC-AUC:         {leaky_auc:.3f}")
print(f"Absolute jump:         {leaky_auc - clean_auc:+.3f}")

assert leaky_auc > clean_auc
assert leaky_auc > 0.98
print("PASS: direct label leakage produced the expected near-perfect score.")


Clean-feature ROC-AUC: 0.634
Leaky ROC-AUC:         1.000
Absolute jump:         +0.366
PASS: direct label leakage produced the expected near-perfect score.


## 7. Leakage attack B — future-window feature

`april_impressions` is also forbidden, even though it is not literally the label. It comes from the same future outcome window used to define the label. At the March decision point it does not exist yet.

This demonstrates the second major leakage pattern: a feature can leak the answer simply by overlapping the future label window.


In [8]:
future_leak = model_df.copy()
future_leak["april_impressions"] = feature_frame["april_impressions"]

Xf = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(
        future_leak[clean_features + ["april_impressions"]]
    ),
    columns=clean_features + ["april_impressions"],
    index=future_leak.index,
)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    Xf, y, test_size=0.20, random_state=42, stratify=y
)
future_auc = fit_auc(Xf_train, Xf_test, yf_train, yf_test)

print(f"Clean-feature ROC-AUC:      {clean_auc:.3f}")
print(f"Future-window ROC-AUC:      {future_auc:.3f}")
print(f"Absolute jump from future: {future_auc - clean_auc:+.3f}")
print("Verdict: april_impressions is excluded because it is unavailable at the March decision point.")
assert "april_impressions" not in clean_features


Clean-feature ROC-AUC:      0.634
Future-window ROC-AUC:      1.000
Absolute jump from future: +0.366
Verdict: april_impressions is excluded because it is unavailable at the March decision point.


## 8. Final feature firewall

The final feature list is the only list eligible for downstream modeling. IDs are context, April fields are future information, and the label itself is the target—not a feature.


In [9]:
FINAL_FEATURES = clean_features.copy()

FORBIDDEN_FEATURES = {
    "future_decline_label",
    "april_impressions",
    "client_hash_id",
    "content_hash_id",
}

assert len(FINAL_FEATURES) == 5
assert not set(FINAL_FEATURES) & FORBIDDEN_FEATURES
assert "leaked_decline_score" not in FINAL_FEATURES

final_frame = feature_frame[
    ["client_hash_id", "content_hash_id"] + FINAL_FEATURES + ["future_decline_label"]
].copy()

print("Final feature set:")
for i, feature in enumerate(FINAL_FEATURES, 1):
    print(f"{i}. {feature}")
print(f"\nFeature count: {len(FINAL_FEATURES)}")
print(f"Forbidden columns present in final feature list: {sorted(set(FINAL_FEATURES) & FORBIDDEN_FEATURES)}")
print("PASS: final feature vector contains only five March decision-time-safe features.")


Final feature set:
1. march_impressions
2. march_clicks
3. march_ctr_pct
4. march_avg_position
5. march_impression_days

Feature count: 5
Forbidden columns present in final feature list: []
PASS: final feature vector contains only five March decision-time-safe features.


## 9. Limitations

- The warehouse is an unbalanced panel, so clients do not all have the same history depth.
- GSC availability is not universal; unavailable rows are excluded using `IS TRUE` rather than interpreted as observed zero activity.
- The April decline label is an observed outcome proxy, not a causal explanation or business decision.
- A single March→April transition cannot establish long-run stability or seasonality.
- The five-feature vector intentionally captures only a small set of search-performance signals; it is not a complete representation of content quality or business impact.
- The quick ROC-AUC experiments use a random train/test split only as a leakage demonstration. They are not a claim of final deployment performance; later modeling should use time-aware and/or grouped validation.


## 10. Self-check

This optional notebook is complete when the assertions below pass and the outputs visibly show the leakage jump and its removal.


In [10]:
assert grain_check.empty
assert str(window_check.loc[0, "min_report_date"])[:10] == "2026-03-01"
assert str(window_check.loc[0, "max_report_date"])[:10] == "2026-03-31"
assert int(availability.loc[0, "gsc_available_rows"]) > 0
assert len(FINAL_FEATURES) == 5
assert leaky_auc > 0.98
assert "april_impressions" not in FINAL_FEATURES
assert "leaked_decline_score" not in FINAL_FEATURES
assert not set(FINAL_FEATURES) & FORBIDDEN_FEATURES

print("SELF-CHECK: PASS")
print("• Raw grain verified")
print("• March decision window verified")
print("• Availability checked with IS TRUE")
print("• Five decision-time-safe features retained")
print("• Direct label leakage demonstrated")
print("• Future-window leakage demonstrated")
print("• Leaky columns excluded from final feature vector")
print("• Limitations stated")
print("• No credentials are stored in notebook source")


SELF-CHECK: PASS
• Raw grain verified
• March decision window verified
• Availability checked with IS TRUE
• Five decision-time-safe features retained
• Direct label leakage demonstrated
• Future-window leakage demonstrated
• Leaky columns excluded from final feature vector
• Limitations stated
• No credentials are stored in notebook source
